In [ ]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from data.DataLoader import DataLoader
from data.Elastic import Elastic

from MultiLayerPerceptron import MLP
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
dataloader = DataLoader(
    Elastic(timeout=5), 
    index='lexical'
)

In [ ]:
X = dataloader.df.drop(['url', 'type'], axis=1, inplace=False)
y = dataloader.df['type']

X_train, X_test, y_train, y_test = dataloader.train_test_split(X, y, train_split=0.8, random_state=None)

In [ ]:
input_size = X_train.shape[1]
model = MLP(input_size=100, hidden_layers=[128, 64, 32], output_size=10, activation_fn=nn.ReLU, dropout_rate=0.5)

_hyperparams = {
    'lr': 0.01, 
    'momentum': 0,
    'dampening': 0,
    'weight_decay': 0,
    'nesterov': False
}

criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), **_hyperparams)

num_epochs = 5000

In [ ]:
X_train_tensor = torch.from_numpy(X_train.to_numpy().astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.to_numpy().astype(np.float32))

X_test_tensor = torch.from_numpy(X_test.to_numpy().astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.to_numpy().astype(np.float32))

In [ ]:
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor.view(-1, 1))
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

In [ ]:
model.eval()
with torch.no_grad():
    predictions = model(X_test_tensor)
    predictions = (predictions > 0.5).float()

accuracy = (predictions == y_test_tensor.view(-1, 1)).sum().item() / len(y_test)
print(f'Test Accuracy: {accuracy * 100:.2f}%')

In [ ]:
# Export the model
torch.onnx.export(model,               # model being run
    X_train_tensor,                         # model input (or a tuple for multiple inputs)
    "mlp_model.onnx",   # where to save the model (can be a file or file-like object)
    export_params=True,        # store the trained parameter weights inside the model file
    #opset_version=10,          # the ONNX version to export the model to
    #do_constant_folding=True,  # whether to execute constant folding for optimization
    input_names = ['input'],   # the model's input names
    output_names = ['output'], # the model's output names
    dynamic_axes={'input' : {0 : 'batch_size'},    # variable length axes
                'output' : {0 : 'batch_size'}})